# CREAM Progressive Graph Noise

Loads progressive graph-noise runs and plots task accuracy, concept accuracy, CCI, PFI, and intervention curves as a function of graph distance.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'experiments').exists() and (PROJECT_ROOT.parent / 'experiments').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

dataset_roots = {
    'cub': PROJECT_ROOT / 'experiments/CUB/train_cbm/Standard_CUB/cub_progressive_noise/CREAM_cub_progressive_noise',
}
metadata_paths = {
    'cub': PROJECT_ROOT / 'data/CUB/progressive_noise_graph/progressive_noise_metadata.csv',
}
summary_out = PROJECT_ROOT / 'notebook/progressive_noise_summary_values.csv'
intervention_out = PROJECT_ROOT / 'notebook/progressive_noise_intervention_values.csv'

def load_last_metrics(dataset_roots, metadata_paths):
    rows = []
    for dataset, root in dataset_roots.items():
        for csv_path in sorted(root.glob('noise_*/last_metrics/*.csv')):
            df = pd.read_csv(csv_path)
            if df.empty:
                continue
            row = df.iloc[0].to_dict()
            row['dataset'] = dataset
            row['noise_percent'] = int(csv_path.relative_to(root).parts[0].replace('noise_', ''))
            row['csv_path'] = str(csv_path)
            rows.append(row)
    results = pd.DataFrame(rows)
    metadata_rows = []
    for dataset, path in metadata_paths.items():
        if path.exists():
            meta = pd.read_csv(path)
            if 'dataset' not in meta.columns:
                meta['dataset'] = dataset
            metadata_rows.append(meta)
    metadata = pd.concat(metadata_rows, ignore_index=True) if metadata_rows else pd.DataFrame()
    if results.empty:
        return results
    if not metadata.empty:
        results = results.merge(metadata, on=['dataset', 'noise_percent'], how='left', suffixes=('', '_metadata'))
    if 'graph_distance' not in results.columns:
        results['graph_distance'] = results['noise_percent'] / 100.0
    return results.sort_values(['dataset', 'noise_percent'])

def load_interventions(dataset_roots, metadata_paths):
    rows = []
    for dataset, root in dataset_roots.items():
        for csv_path in sorted(root.glob('noise_*/lightning_logs/**/intervention_results.csv')):
            df = pd.read_csv(csv_path)
            if df.empty:
                continue
            df['dataset'] = dataset
            df['noise_percent'] = int(csv_path.relative_to(root).parts[0].replace('noise_', ''))
            df['csv_path'] = str(csv_path)
            rows.append(df)
    interventions = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    if interventions.empty:
        return interventions
    metadata_rows = []
    for dataset, path in metadata_paths.items():
        if path.exists():
            meta = pd.read_csv(path)
            if 'dataset' not in meta.columns:
                meta['dataset'] = dataset
            metadata_rows.append(meta)
    metadata = pd.concat(metadata_rows, ignore_index=True) if metadata_rows else pd.DataFrame()
    if not metadata.empty:
        interventions = interventions.merge(metadata, on=['dataset', 'noise_percent'], how='left', suffixes=('', '_metadata'))
    if 'graph_distance' not in interventions.columns:
        interventions['graph_distance'] = interventions['noise_percent'] / 100.0
    return interventions.sort_values(['dataset', 'noise_percent'])

def intervention_accuracy_column(df):
    for col in ['test_task_accuracy', 'task_accuracy', 'accuracy', 'test_acc']:
        if col in df.columns:
            return col
    matches = [col for col in df.columns if 'accuracy' in col.lower() or col.lower().endswith('_acc')]
    return matches[0] if matches else None

def available(columns, candidates):
    return [col for col in candidates if col and col in columns]


## Saved Values Table

In [ ]:
merged = load_last_metrics(dataset_roots, metadata_paths)

value_cols = available(
    merged.columns,
    [
        'dataset', 'noise_percent', 'graph_distance',
        'num_flipped_positions', 'different_edges', 'added_edges', 'deleted_edges',
        'original_true_entries', 'perturbed_true_entries', 'perturbed_edges',
        'test_task_accuracy', 'test_concept_accuracy', 'test_dropout_task_accuracy',
        'CCI', 'PFI_concept_importance', 'PFI_side_importance',
        'csv_path',
    ],
)

if merged.empty:
    print('No progressive-noise result CSVs found yet. Run the progressive_noise Condor jobs first.')
else:
    summary_values = merged[value_cols].sort_values(['dataset', 'noise_percent'])
    summary_values.to_csv(summary_out, index=False)
    print(f'Saved summary values to {summary_out}')
    display(summary_values)


## Accuracy, CCI, and PFI Curves

In [ ]:
metrics_to_plot = [
    ('test_task_accuracy', 'Task accuracy'),
    ('test_concept_accuracy', 'Concept accuracy'),
    ('CCI', 'CCI'),
    ('PFI_concept_importance', 'PFI concept importance'),
    ('PFI_side_importance', 'PFI side importance'),
]

if merged.empty:
    print('No progressive-noise metrics available yet.')
else:
    x_col = 'graph_distance' if 'graph_distance' in merged.columns else 'noise_percent'
    for metric, label in metrics_to_plot:
        if metric not in merged.columns:
            print(f'Skipping {metric}: column not found in last_metrics.')
            continue
        for dataset, df in merged.groupby('dataset'):
            df = df.dropna(subset=[x_col, metric]).sort_values(x_col)
            if df.empty:
                continue
            fig, ax = plt.subplots(figsize=(7, 4))
            ax.plot(df[x_col], df[metric], marker='o', linewidth=2)
            ax.set_xlabel('Graph distance = different edge positions / total positions' if x_col == 'graph_distance' else 'Noise percent')
            ax.set_ylabel(label)
            ax.set_title(f'{dataset}: {label} vs progressive graph noise')
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()


## Combined Metric Curves

In [ ]:
if merged.empty:
    print('No progressive-noise metrics available yet.')
else:
    x_col = 'graph_distance' if 'graph_distance' in merged.columns else 'noise_percent'
    for dataset, df in merged.groupby('dataset'):
        df = df.sort_values(x_col)
        plot_cols = [metric for metric, _ in metrics_to_plot if metric in df.columns and df[metric].notna().any()]
        if not plot_cols:
            print(f'No plottable metric columns for {dataset}.')
            continue
        fig, ax = plt.subplots(figsize=(8, 4.5))
        for metric in plot_cols:
            ax.plot(df[x_col], df[metric], marker='o', linewidth=2, label=metric)
        ax.set_xlabel('Graph distance = different edge positions / total positions' if x_col == 'graph_distance' else 'Noise percent')
        ax.set_ylabel('Value')
        ax.set_title(f'{dataset}: progressive graph-noise metrics')
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()


## Intervention Values and Curves

In [ ]:
interventions = load_interventions(dataset_roots, metadata_paths)
acc_col = intervention_accuracy_column(interventions)

intervention_cols = available(
    interventions.columns,
    ['dataset', 'noise_percent', 'graph_distance', 'group_interventions', 'num_interventions', acc_col, 'csv_path'],
)

if interventions.empty:
    print('No intervention_results.csv files found yet. They are written under noise_*/lightning_logs/**/ after evaluation.')
else:
    intervention_values = interventions[intervention_cols].sort_values(['dataset', 'noise_percent', 'group_interventions', 'num_interventions'])
    intervention_values.to_csv(intervention_out, index=False)
    print(f'Saved intervention values to {intervention_out}')
    display(intervention_values)


In [ ]:
if interventions.empty or acc_col is None or 'num_interventions' not in interventions.columns:
    print('No plottable intervention curves yet.')
else:
    for dataset, dataset_df in interventions.groupby('dataset'):
        group_values = [None]
        if 'group_interventions' in dataset_df.columns:
            group_values = sorted(dataset_df['group_interventions'].dropna().unique())
        for group_value in group_values:
            df = dataset_df if group_value is None else dataset_df[dataset_df['group_interventions'] == group_value]
            if df.empty:
                continue
            hue_col = 'graph_distance' if 'graph_distance' in df.columns else 'noise_percent'
            fig, ax = plt.subplots(figsize=(8, 5))
            for hue_value, hue_df in df.groupby(hue_col):
                hue_df = hue_df.sort_values('num_interventions')
                ax.plot(hue_df['num_interventions'], hue_df[acc_col], marker='o', linewidth=2, label=str(hue_value))
            suffix = '' if group_value is None else f' | group_interventions={group_value}'
            ax.set_xlabel('Number of interventions')
            ax.set_ylabel(acc_col)
            ax.set_title(f'{dataset}: intervention curve under progressive graph noise{suffix}')
            ax.grid(True, alpha=0.3)
            ax.legend(title=hue_col, bbox_to_anchor=(1.02, 1), loc='upper left')
            plt.tight_layout()
            plt.show()
